In [1]:
from keras.models import Model 
from keras.layers import Input, Convolution2D, MaxPooling2D, Dense, Dropout, Flatten, Dense, Dropout, Flatten, Conv2D, MaxPooling2D
# import np_utils
import numpy as np
import pandas as pd
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import os
import datetime
from sklearn.model_selection import train_test_split

from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import SpectralClustering


from sklearn.utils import resample
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import pairwise_distances_argmin_min
from tslearn.metrics import cdist_dtw
from sklearn.metrics import silhouette_score
from tslearn.clustering import silhouette_score

import tensorflow as tf

In [21]:
RANDOM_STATE = 42
sil_sample=20000

# Define functions

In [3]:
def agglom_dtw(embeds, k, distance):

    embeds_3D = np.expand_dims(embeds, axis=-1)
    sample_size = 1000  
    sampled_embeds = resample(embeds_3D, n_samples=sample_size, random_state=42)

    dtw_distance_matrix = cdist_dtw(sampled_embeds)

    hc = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    sampled_labels = hc.fit_predict(dtw_distance_matrix)

 
    cluster_centroids = []
    for i in range(k):
        cluster_indices = np.where(sampled_labels == i)[0]
        cluster_series = sampled_embeds[cluster_indices]
        dist_matrix_cluster = cdist_dtw(cluster_series)
        medoid_idx_in_cluster = np.argmin(dist_matrix_cluster.sum(axis=1))
        cluster_centroids.append(cluster_series[medoid_idx_in_cluster])
        
    cluster_centroids = np.array(cluster_centroids)  
    
    from tslearn.clustering import TimeSeriesKMeans

    ts_kmeans = TimeSeriesKMeans(
        n_clusters=k,
        metric="dtw",
        max_iter=0,
        random_state=42
    )
    ts_kmeans.cluster_centers_ = cluster_centroids

    labels = ts_kmeans.predict(embeds_3D)
    
    return labels


In [4]:
def agglomerative(embeds, k, distance, assign_method):

    sample_size = 5000 
    sampled_embeds = resample(embeds, n_samples=sample_size, random_state=42)
    
    hierarchical_cluster = AgglomerativeClustering(n_clusters=k, metric=distance, linkage='ward')
    labels_sub = hierarchical_cluster.fit_predict(sampled_embeds)
    
    if (assign_method == 'knn'): 
        neigh = KNeighborsClassifier(n_neighbors=13)
        neigh.fit(sampled_embeds, labels_sub)
        labels = neigh.predict(embeds) 
    else:      
        centroids_model = NearestCentroid()
        centroids_model.fit(sampled_embeds, labels_sub)
        cluster_centroids = centroids_model.centroids_
        labels, _  = pairwise_distances_argmin_min(embeds, cluster_centroids)

    return labels

In [5]:
def sil_score(X_embed, labels, s, debug=1): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="euclidean")
    else:
        sil = silhouette_score(X_embed, labels, metric="euclidean")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [6]:
def sil_score_dtw(X_embed, labels, s, debug=1): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="dtw")
    else:
        sil = silhouette_score(X_embed, labels, metric="dtw")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [7]:
def DAE_reduction(df, bottleneck, ep, batch):
    df_flat = [sample.flatten() for sample in df]
    df_flat = np.asarray(df_flat)
    train, test = train_test_split(df_flat, test_size=0.20, random_state=42)
    input = Input(shape=(df_flat.shape[1],))

    encoded = Dense(30, activation='relu')(input)
    encoded = Dense(20, activation='relu')(encoded)
    encoded = Dense(10, activation='relu')(encoded)
    encoded = Dense(bottleneck, activation='linear', name='bottleneck')(encoded)

    decoded = Dense(bottleneck, activation='relu')(encoded)
    decoded = Dense(20, activation='relu')(decoded)
    decoded = Dense(30, activation='relu')(decoded)
    output = Dense(df_flat.shape[1], activation=None)(decoded)

    autoencoder = Model(input, output)
#   autoencoder.summary()
#   autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    huber = tf.keras.losses.Huber()
    autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss=huber)
    autoencoder.fit(train, train,
     epochs=ep,
     batch_size=batch,
     shuffle=True,
     validation_data=(test, test), verbose = 0)
    encoder = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer('bottleneck').output)
    encoded_ts = encoder.predict(df_flat)
    return encoded_ts, encoder 

In [9]:
# Hyperparameter tuning; 
# hardcoded for DAE with linear bottleneck layer, 2048 batch size and Huber loss; 
# may want to tune them as well

def hypertune(target, k, clustering, abm): 

    BOTTLENECK_MIN = 5 
    EPOCH_MIN = 3
    
    bottleneck = 5     #  number of desired embeddings; tune in the range of [BOTTLENECK_MIN..bottleneck]
    epochs = 3         #  tune in range of [EPOCH_MIN..epochs] 
    batch = 2048       #  fixed from prior trials 
    evals = 10           #  number of repeated trainings to reduce noise in model 
  
    sil_sample = 20000 # silhouette sample 

    max_sil_score = 0         # max silohouette score 
    best_embed = []           # best emebeddings 
    best_model = []           # model that produced the best embeddings 
    
    for b in np.arange(BOTTLENECK_MIN, bottleneck + 1):   # bottleneck 
        for e in np.arange(EPOCH_MIN, epochs + 1):        # epoch
            for i in np.arange(1, evals + 1):             # evals 
                
                print(f"Evaluation: embeddings {b}, epoch {e}, eval # {i}:")
                # get embeddings 
                dae_embeds, dae_model = DAE_reduction(raw_data[target], bottleneck, epochs, batch)

                # clustering 
                if (clustering == 'agglom-dtw'):
                 #   dae_labels = agglomerative(dae_embeds, k, 'euclidean', 'knn')
                    dae_labels = agglom_dtw(dae_embeds, k, 'dtw')
                elif (clustering == 'agglom-euc'):
                    dae_labels = agglomerative(dae_embeds, k, 'euclidean', 'knn')
                    #dae_labels = agglom_dtw(dae_embeds, k, 'dtw')
                else: 
                    dae_labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit(dae_embeds).labels_
                
                unique_labels = np.unique(dae_labels)
                if (len(unique_labels) > 1): 
                    # silhouette score 
                    score = sil_score(dae_embeds, dae_labels, sil_sample)
                    # save the best embeddings and model   
                    if (score > max_sil_score): 
                        max_sil_score = score
                        best_embed = dae_embeds
                        best_model.append(dae_model) # assuming we are hypertuning the three models in order  
                        best_params = str(b) + '_' + str(e) + '_' + str(batch) + '_linear'
                    # log result 
                    outfile = 'dae_sil_scores_all_dtw_' + clustering + '.csv'
                    with open(outfile, "a") as f:
                        f.write(f"{abm},{k},{b},{e},{i},{score:.4f}\n")
                else: 
                    print(f"Error: Too few labels generated. Skipping ...")
 
    print(f"Best Silhouette: {max_sil_score:.4f}")
    print(f"Best parameters: {best_params}")
    filename = 'extracted_features/' + str(target) + '_DAE_' + best_params + '_' + clustering + '.csv'
    np.savetxt(filename, best_embed, delimiter=",")
    with open("daep_sil_scores.csv", "a") as f:
        f.write(f"{abm},{k},{max_sil_score:.4f}\n")

In [10]:
def import_ff_data(filename):
    expected_columns=155
    data = []
    with open(filename, 'r') as file:
        for line in file:
            row = line.strip().split(',')
            if len(row) < expected_columns:
                row += [np.nan] * (expected_columns - len(row))
            data.append(row)
    df = pd.DataFrame(data)
    def fill_last_valid(row):
        for i in range(1, len(row)):
            if pd.isna(row[i]):
                row[i] = row[i - 1]  
        return row
    df_filled = df.apply(fill_last_valid, axis=1)
    return df_filled

# Import Data

In [11]:
poor = pd.read_csv("SimData/bank_reserves_outputs_poor.csv", header=None)
middle = pd.read_csv("SimData/bank_reserves_outputs_middle.csv", header=None)
rich = pd.read_csv("SimData/bank_reserves_outputs_rich.csv", header=None)
sc = StandardScaler()
br = []
for i in np.arange(0, poor.shape[0]):
    sample = pd.concat([poor.iloc[i], middle.iloc[i]], axis=0).T
    sample = pd.concat([sample, rich.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    br.append(sample_std)

In [12]:
ecv_active = pd.read_csv("SimData/epsteinCV_outputs_active.csv", header=None)
ecv_jailed = pd.read_csv("SimData/epsteinCV_outputs_jailed.csv", header=None)
ecv_quiet = pd.read_csv("SimData/epsteinCV_outputs_quiet.csv", header=None)
sc = StandardScaler()
ecv = []
for i in np.arange(0, ecv_active.shape[0]):
    sample = pd.concat([ecv_active.iloc[i], ecv_jailed.iloc[i]], axis=0).T
    sample = pd.concat([sample, ecv_quiet.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ecv.append(sample_std)

In [13]:
ff_onfire = import_ff_data("SimData/forest_fire_outputs_onfire.csv")
print("check 1")
ff_fine = import_ff_data("SimData/forest_fire_outputs_fine.csv")
print("check 2")
ff_burned = import_ff_data("SimData/forest_fire_outputs_burned.csv")
sc = StandardScaler()
ff = []
for i in np.arange(0, ff_onfire.shape[0]):
    sample = pd.concat([ff_onfire.iloc[i], ff_fine.iloc[i]], axis=0).T
    sample = pd.concat([sample, ff_burned.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ff.append(sample_std)

check 1
check 2


In [14]:
ABMs = ["BR", "ECV", "FF"]
raw_data = []
raw_data.append(br)
raw_data.append(ecv)
raw_data.append(ff)

In [15]:
# Extract PCA embeddings; not re-doing PCA in this notebook
pca_br_embeds = pd.read_csv('extracted_features/bank_reserves_pca_standardized.csv')
pca_br_embeds = pca_br_embeds.drop('Unnamed: 0', axis=1)
pca_br_embeds = pca_br_embeds.to_numpy()

pca_ecv_embeds = pd.read_csv('extracted_features/epstein_pca_standardized.csv')
pca_ecv_embeds = pca_ecv_embeds.drop('Unnamed: 0', axis=1)
pca_ecv_embeds = pca_ecv_embeds.to_numpy()

pca_ff_embeds = pd.read_csv('extracted_features/forestfire_pca_standardized.csv')
pca_ff_embeds = pca_ff_embeds.drop('Unnamed: 0', axis=1)
pca_ff_embeds = pca_ff_embeds.to_numpy()

pca_embeds = []
pca_embeds.append(pca_br_embeds)
pca_embeds.append(pca_ecv_embeds)
pca_embeds.append(pca_ff_embeds)

In [16]:
# Extract DAE embeddings; not re-doing DAE in this notebook
dae_br_embeds = pd.read_csv('extracted_features/bank_reserves_dae.csv')
#dae_br_embeds = dae_br_embeds.drop('Unnamed: 0', axis=1)
dae_br_embeds = dae_br_embeds.to_numpy()

dae_ecv_embeds = pd.read_csv('extracted_features/epstein_dae.csv')
#dae_ecv_embeds = dae_ecv_embeds.drop('Unnamed: 0', axis=1)
dae_ecv_embeds = dae_ecv_embeds.to_numpy()

dae_ff_embeds = pd.read_csv('extracted_features/forestfire_dae.csv')
#dae_ff_embeds = dae_ff_embeds.drop('Unnamed: 0', axis=1)
dae_ff_embeds = dae_ff_embeds.to_numpy()

dae_embeds = []
dae_embeds.append(dae_br_embeds)
dae_embeds.append(dae_ecv_embeds)
dae_embeds.append(dae_ff_embeds)

In [17]:
# Extract DCAE embeddings; not re-doing DCAE in this notebook
dcae_br_embeds = pd.read_csv('extracted_features/bank_reserves_dcae.csv')
#dcae_br_embeds = dcae_br_embeds.drop('Unnamed: 0', axis=1)
dcae_br_embeds = dcae_br_embeds.to_numpy()

dcae_ecv_embeds = pd.read_csv('extracted_features/epstein_dcae.csv')
#dcae_ecv_embeds = dcae_ecv_embeds.drop('Unnamed: 0', axis=1)
dcae_ecv_embeds = dcae_ecv_embeds.to_numpy()

dcae_ff_embeds = pd.read_csv('extracted_features/forestfire_dcae.csv')
#dcae_ff_embeds = dcae_ff_embeds.drop('Unnamed: 0', axis=1)
dcae_ff_embeds = dcae_ff_embeds.to_numpy()

dcae_embeds = []
dcae_embeds.append(dcae_br_embeds)
dcae_embeds.append(dcae_ecv_embeds)
dcae_embeds.append(dcae_ff_embeds)

In [18]:
# Extract DAEP embeddings; not re-doing DAEP in this notebook
daep_br_embeds = pd.read_csv('extracted_features/0_DAE_5_2_2048_linear.csv')
#daep_br_embeds = daep_br_embeds.drop('Unnamed: 0', axis=1)
daep_br_embeds = daep_br_embeds.to_numpy()

daep_ecv_embeds = pd.read_csv('extracted_features/1_DAE_5_3_2048_linear.csv')
#daep_ecv_embeds = daep_ecv_embeds.drop('Unnamed: 0', axis=1)
daep_ecv_embeds = daep_ecv_embeds.to_numpy()

daep_ff_embeds = pd.read_csv('extracted_features/2_DAE_3_3_2048_linear.csv')
#daep_ff_embeds = daep_ff_embeds.drop('Unnamed: 0', axis=1)
daep_ff_embeds = daep_ff_embeds.to_numpy()

daep_embeds = []
daep_embeds.append(daep_br_embeds)
daep_embeds.append(daep_ecv_embeds)
daep_embeds.append(daep_ff_embeds)

In [19]:
opt_clusters = []
opt_clusters.append(7)
opt_clusters.append(8)
opt_clusters.append(4)

# Cluster Raw Data

In [22]:
# Get silhouette scores for raw features
with open("raw_sil_scores.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        flat_list = [sample.flatten() for sample in raw_data[i]]
        raw_features = np.asarray(flat_list)
        raw_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(raw_features).labels_
        unique_labels = np.unique(raw_labels)
        if (len(unique_labels) > 1): 
            score = sil_score(raw_features, raw_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[Raw data,{abm},{k}] \t Silhouette Score: {score:.4f}")  
        with open("raw_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}, {raw_labels} \n")

Silhouette Score: 0.6218
[Raw data,BR,3] 	 Silhouette Score: 0.6218
Silhouette Score: 0.5445
[Raw data,BR,4] 	 Silhouette Score: 0.5445
Silhouette Score: 0.4739
[Raw data,BR,5] 	 Silhouette Score: 0.4739
Silhouette Score: 0.3163
[Raw data,BR,6] 	 Silhouette Score: 0.3163
Silhouette Score: 0.2885
[Raw data,BR,7] 	 Silhouette Score: 0.2885
Silhouette Score: 0.2560
[Raw data,BR,8] 	 Silhouette Score: 0.2560
Silhouette Score: 0.2364
[Raw data,BR,9] 	 Silhouette Score: 0.2364
Silhouette Score: 0.2132
[Raw data,BR,10] 	 Silhouette Score: 0.2132
Silhouette Score: 0.6034
[Raw data,ECV,3] 	 Silhouette Score: 0.6034
Silhouette Score: 0.5946
[Raw data,ECV,4] 	 Silhouette Score: 0.5946
Silhouette Score: 0.5840
[Raw data,ECV,5] 	 Silhouette Score: 0.5840
Silhouette Score: 0.5898
[Raw data,ECV,6] 	 Silhouette Score: 0.5898
Silhouette Score: 0.5281
[Raw data,ECV,7] 	 Silhouette Score: 0.5281
Silhouette Score: 0.5251
[Raw data,ECV,8] 	 Silhouette Score: 0.5251
Silhouette Score: 0.5175
[Raw data,ECV,9]

# Kmean Cluster Reference Methods (PCA, DAE, DCAE)

In [23]:
# Get silhouette scores for pca features 
with open("pca_sil_scores.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs):
    for k in range(3,11):
        pca_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(pca_embeds[i]).labels_
        unique_labels = np.unique(pca_labels)
        if (len(unique_labels) > 1): 
            score = sil_score(pca_embeds[i], pca_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[PCA,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("pca_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}, {pca_labels} \n")

Silhouette Score: 0.7625
[PCA,BR,3] 	 Silhouette Score: 0.7625
Silhouette Score: 0.6950
[PCA,BR,4] 	 Silhouette Score: 0.6950
Silhouette Score: 0.6041
[PCA,BR,5] 	 Silhouette Score: 0.6041
Silhouette Score: 0.5890
[PCA,BR,6] 	 Silhouette Score: 0.5890
Silhouette Score: 0.5566
[PCA,BR,7] 	 Silhouette Score: 0.5566
Silhouette Score: 0.5487
[PCA,BR,8] 	 Silhouette Score: 0.5487
Silhouette Score: 0.5370
[PCA,BR,9] 	 Silhouette Score: 0.5370
Silhouette Score: 0.5243
[PCA,BR,10] 	 Silhouette Score: 0.5243
Silhouette Score: 0.6228
[PCA,ECV,3] 	 Silhouette Score: 0.6228
Silhouette Score: 0.6210
[PCA,ECV,4] 	 Silhouette Score: 0.6210
Silhouette Score: 0.6165
[PCA,ECV,5] 	 Silhouette Score: 0.6165
Silhouette Score: 0.6207
[PCA,ECV,6] 	 Silhouette Score: 0.6207
Silhouette Score: 0.5668
[PCA,ECV,7] 	 Silhouette Score: 0.5668
Silhouette Score: 0.5704
[PCA,ECV,8] 	 Silhouette Score: 0.5704
Silhouette Score: 0.5679
[PCA,ECV,9] 	 Silhouette Score: 0.5679
Silhouette Score: 0.5691
[PCA,ECV,10] 	 Silhoue

In [24]:
# Get silhouette scores for dae features 
with open("dae_sil_scores.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        dae_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(dae_embeds[i]).labels_
        unique_labels = np.unique(dae_labels)
        if (len(unique_labels) > 1): 
            score = sil_score(dae_embeds[i], dae_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[dae,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("dae_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}, {dae_labels} \n")

Silhouette Score: 0.7401
[dae,BR,3] 	 Silhouette Score: 0.7401
Silhouette Score: 0.7110
[dae,BR,4] 	 Silhouette Score: 0.7110
Silhouette Score: 0.6865
[dae,BR,5] 	 Silhouette Score: 0.6865
Silhouette Score: 0.6449
[dae,BR,6] 	 Silhouette Score: 0.6449
Silhouette Score: 0.5739
[dae,BR,7] 	 Silhouette Score: 0.5739
Silhouette Score: 0.5693
[dae,BR,8] 	 Silhouette Score: 0.5693
Silhouette Score: 0.5679
[dae,BR,9] 	 Silhouette Score: 0.5679
Silhouette Score: 0.5617
[dae,BR,10] 	 Silhouette Score: 0.5617
Silhouette Score: 0.5532
[dae,ECV,3] 	 Silhouette Score: 0.5532
Silhouette Score: 0.5724
[dae,ECV,4] 	 Silhouette Score: 0.5724
Silhouette Score: 0.5623
[dae,ECV,5] 	 Silhouette Score: 0.5623
Silhouette Score: 0.5728
[dae,ECV,6] 	 Silhouette Score: 0.5728
Silhouette Score: 0.5675
[dae,ECV,7] 	 Silhouette Score: 0.5675
Silhouette Score: 0.5394
[dae,ECV,8] 	 Silhouette Score: 0.5394
Silhouette Score: 0.5492
[dae,ECV,9] 	 Silhouette Score: 0.5492
Silhouette Score: 0.5536
[dae,ECV,10] 	 Silhoue

In [25]:
# Get silhouette scores for dcae features 
with open("dcae_sil_scores.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        dcae_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(dcae_embeds[i]).labels_
        unique_labels = np.unique(dcae_labels)
        if (len(unique_labels) > 1): 
            score = sil_score(dcae_embeds[i], dcae_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[dcae,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("dcae_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}\n")

Silhouette Score: 0.6009
[dcae,BR,3] 	 Silhouette Score: 0.6009
Silhouette Score: 0.5222
[dcae,BR,4] 	 Silhouette Score: 0.5222
Silhouette Score: 0.4304
[dcae,BR,5] 	 Silhouette Score: 0.4304
Silhouette Score: 0.4249
[dcae,BR,6] 	 Silhouette Score: 0.4249
Silhouette Score: 0.4102
[dcae,BR,7] 	 Silhouette Score: 0.4102
Silhouette Score: 0.3883
[dcae,BR,8] 	 Silhouette Score: 0.3883
Silhouette Score: 0.3762
[dcae,BR,9] 	 Silhouette Score: 0.3762
Silhouette Score: 0.3672
[dcae,BR,10] 	 Silhouette Score: 0.3672
Silhouette Score: 0.5650
[dcae,ECV,3] 	 Silhouette Score: 0.5650
Silhouette Score: 0.5731
[dcae,ECV,4] 	 Silhouette Score: 0.5731
Silhouette Score: 0.5605
[dcae,ECV,5] 	 Silhouette Score: 0.5605
Silhouette Score: 0.5624
[dcae,ECV,6] 	 Silhouette Score: 0.5624
Silhouette Score: 0.5054
[dcae,ECV,7] 	 Silhouette Score: 0.5054
Silhouette Score: 0.5254
[dcae,ECV,8] 	 Silhouette Score: 0.5254
Silhouette Score: 0.5310
[dcae,ECV,9] 	 Silhouette Score: 0.5310
Silhouette Score: 0.5244
[dcae,E

# Kmeans DAE Plus

In [26]:
# Get silhouette scores for dcae features 
with open("daep_sil_scores.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        daep_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(daep_embeds[i]).labels_
        unique_labels = np.unique(daep_labels)
        if (len(unique_labels) > 1): 
            score = sil_score(daep_embeds[i], daep_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[daep,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("daep_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}, {daep_labels} \n")

Silhouette Score: 0.7635
[daep,BR,3] 	 Silhouette Score: 0.7635
Silhouette Score: 0.7412
[daep,BR,4] 	 Silhouette Score: 0.7412
Silhouette Score: 0.7204
[daep,BR,5] 	 Silhouette Score: 0.7204
Silhouette Score: 0.6984
[daep,BR,6] 	 Silhouette Score: 0.6984
Silhouette Score: 0.6669
[daep,BR,7] 	 Silhouette Score: 0.6669
Silhouette Score: 0.6131
[daep,BR,8] 	 Silhouette Score: 0.6131
Silhouette Score: 0.5793
[daep,BR,9] 	 Silhouette Score: 0.5793
Silhouette Score: 0.5688
[daep,BR,10] 	 Silhouette Score: 0.5688
Silhouette Score: 0.8914
[daep,ECV,3] 	 Silhouette Score: 0.8914
Silhouette Score: 0.8701
[daep,ECV,4] 	 Silhouette Score: 0.8701
Silhouette Score: 0.8293
[daep,ECV,5] 	 Silhouette Score: 0.8293
Silhouette Score: 0.8302
[daep,ECV,6] 	 Silhouette Score: 0.8302
Silhouette Score: 0.8213
[daep,ECV,7] 	 Silhouette Score: 0.8213
Silhouette Score: 0.8207
[daep,ECV,8] 	 Silhouette Score: 0.8207
Silhouette Score: 0.6682
[daep,ECV,9] 	 Silhouette Score: 0.6682
Silhouette Score: 0.6697
[daep,E

# Agglom-KNN DAE Plus

In [27]:
with open("daep_sil_scores_agglom_knn.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        daep_labels = agglomerative(daep_embeds[i], k, "euclidean", 'knn')
        #daep_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(daep_embeds[i]).labels_
        unique_labels = np.unique(daep_labels)
        if (len(unique_labels) > 1):  
            score = sil_score(daep_embeds[i], daep_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[daep,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("daep_sil_scores_agglom_knn.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}, {daep_labels}\n")

Silhouette Score: 0.7543
[daep,BR,3] 	 Silhouette Score: 0.7543
Silhouette Score: 0.7311
[daep,BR,4] 	 Silhouette Score: 0.7311
Silhouette Score: 0.7133
[daep,BR,5] 	 Silhouette Score: 0.7133
Silhouette Score: 0.6311
[daep,BR,6] 	 Silhouette Score: 0.6311
Silhouette Score: 0.6251
[daep,BR,7] 	 Silhouette Score: 0.6251
Silhouette Score: 0.6298
[daep,BR,8] 	 Silhouette Score: 0.6298
Silhouette Score: 0.5916
[daep,BR,9] 	 Silhouette Score: 0.5916
Silhouette Score: 0.5485
[daep,BR,10] 	 Silhouette Score: 0.5485
Silhouette Score: 0.8621
[daep,ECV,3] 	 Silhouette Score: 0.8621
Silhouette Score: 0.8391
[daep,ECV,4] 	 Silhouette Score: 0.8391
Silhouette Score: 0.8201
[daep,ECV,5] 	 Silhouette Score: 0.8201
Silhouette Score: 0.8167
[daep,ECV,6] 	 Silhouette Score: 0.8167
Silhouette Score: 0.8193
[daep,ECV,7] 	 Silhouette Score: 0.8193
Silhouette Score: 0.8187
[daep,ECV,8] 	 Silhouette Score: 0.8187
Silhouette Score: 0.8140
[daep,ECV,9] 	 Silhouette Score: 0.8140
Silhouette Score: 0.7933
[daep,E

# Agglom-centroids DAE Plus

In [28]:
with open("daep_sil_scores_agglom_centroids.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        daep_labels = agglom_dtw(daep_embeds[i], k, 1)
        #daep_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(daep_embeds[i]).labels_
        unique_labels = np.unique(daep_labels)
        if (len(unique_labels) > 1):  
            score = sil_score(daep_embeds[i], daep_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[daep,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("daep_sil_scores_agglom_centroids.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}, {daep_labels} \n")

Silhouette Score: 0.7659
[daep,BR,3] 	 Silhouette Score: 0.7659
Silhouette Score: 0.7191
[daep,BR,4] 	 Silhouette Score: 0.7191
Silhouette Score: 0.7128
[daep,BR,5] 	 Silhouette Score: 0.7128
Silhouette Score: 0.6835
[daep,BR,6] 	 Silhouette Score: 0.6835
Silhouette Score: 0.6755
[daep,BR,7] 	 Silhouette Score: 0.6755
Silhouette Score: 0.6700
[daep,BR,8] 	 Silhouette Score: 0.6700
Silhouette Score: 0.6225
[daep,BR,9] 	 Silhouette Score: 0.6225
Silhouette Score: 0.6113
[daep,BR,10] 	 Silhouette Score: 0.6113
Silhouette Score: 0.8921
[daep,ECV,3] 	 Silhouette Score: 0.8921
Silhouette Score: 0.8729
[daep,ECV,4] 	 Silhouette Score: 0.8729
Silhouette Score: 0.8748
[daep,ECV,5] 	 Silhouette Score: 0.8748
Silhouette Score: 0.8638
[daep,ECV,6] 	 Silhouette Score: 0.8638
Silhouette Score: 0.8638
[daep,ECV,7] 	 Silhouette Score: 0.8638
Silhouette Score: 0.8316
[daep,ECV,8] 	 Silhouette Score: 0.8316
Silhouette Score: 0.8279
[daep,ECV,9] 	 Silhouette Score: 0.8279
Silhouette Score: 0.8307
[daep,E

# Spectral DAE Plus

In [31]:
with open("daep_sil_scores_spectral.csv", "a") as f:
    f.write(f"{"abm"},{"k"},{"score"}, {"labels"} \n")
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        spectral_labs =SpectralClustering(n_clusters=k, assign_labels='discretize', affinity='nearest_neighbors', random_state=0).fit(daep_embeds[i][1:10000]).labels_
        neigh = KNeighborsClassifier(n_neighbors=13)
        neigh.fit(daep_embeds[i][1:10000], spectral_labs)
        daep_labels = neigh.predict(daep_embeds[i])
        unique_labels = np.unique(daep_labels)
        if (len(unique_labels) > 1):  
            score = sil_score(daep_embeds[i], daep_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[daep,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("daep_sil_scores_spectral.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}, {daep_labels} \n")

Silhouette Score: 0.6923
[daep,BR,3] 	 Silhouette Score: 0.6923
Silhouette Score: 0.6908
[daep,BR,4] 	 Silhouette Score: 0.6908
Silhouette Score: 0.6412
[daep,BR,5] 	 Silhouette Score: 0.6412
Silhouette Score: 0.5943
[daep,BR,6] 	 Silhouette Score: 0.5943
Silhouette Score: 0.5324
[daep,BR,7] 	 Silhouette Score: 0.5324
Silhouette Score: 0.4079
[daep,BR,8] 	 Silhouette Score: 0.4079
Silhouette Score: 0.4236
[daep,BR,9] 	 Silhouette Score: 0.4236
Silhouette Score: 0.4323
[daep,BR,10] 	 Silhouette Score: 0.4323
Silhouette Score: 0.0764
[daep,ECV,3] 	 Silhouette Score: 0.0764
Silhouette Score: 0.1021
[daep,ECV,4] 	 Silhouette Score: 0.1021
Silhouette Score: 0.2823
[daep,ECV,5] 	 Silhouette Score: 0.2823
Silhouette Score: 0.3110
[daep,ECV,6] 	 Silhouette Score: 0.3110
Silhouette Score: 0.3732
[daep,ECV,7] 	 Silhouette Score: 0.3732
Silhouette Score: 0.3700
[daep,ECV,8] 	 Silhouette Score: 0.3700
Silhouette Score: 0.3920
[daep,ECV,9] 	 Silhouette Score: 0.3920
Silhouette Score: 0.3914
[daep,E

C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: 0.6788
[daep,FF,3] 	 Silhouette Score: 0.6788


C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: 0.4038
[daep,FF,4] 	 Silhouette Score: 0.4038


...

Silhouette Score: 0.2524
[daep,FF,8] 	 Silhouette Score: 0.2524


C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: 0.1282
[daep,FF,9] 	 Silhouette Score: 0.1282


C:\Users\met48\AppData\Local\anaconda3\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Silhouette Score: 0.1721
[daep,FF,10] 	 Silhouette Score: 0.1721
